# IMPLEMENTATION OF GAUSS ELIMINATION ALGORITHM, LU FACTORIZATION AND GAUSS-JORDAN ELIMINATION

We know the characteristics of all these algorithms, our goal here is to experiment their numerical implementation in python.

## GUASS ELIMATION.

Let's start from a review of all the possible useful operation with arrays in numpy (regarding entries selection and overwriting, but also reshaping).

In [17]:
import numpy as np

A = np.array([
    [1,3,6],
    [4,3,8],
    [5,9,0],
    ], dtype = float)

print(A.shape) # A dimensions

print(A[0,1])   # Element of A in position 0,1 which is first row second column, pos i,j

print(A[0,:])   # Select the first row, equivalently A[0]

print(A[:,1])  # Select the second column, a syntex similar to that above is not possible in the case of columns

print(A[:2,:])  # First two rows, all the columns

print(A[:, :2]) # All rows first two columns

print(A[1:, 2:])    # Rows from index one onward, columns from index 2 onward

# So the general syntax is [i:n, j:m] where i is the first index of the rows of interest that we want to extract, while n-1 is the index of the last
# row (so the n-th row is actually excluded)
# Simingly for the columns. So this gives us the possibility to extract blocks from a given matrix basically

print('\n', A[0:2, 0:2])  #Upper left 2x2 block of A, it is equivalent to say: "exctract all the element in the first 2 rows and first two columns"

B = np.array([
    [1, 2.4, -4, 5.5],
    [6, 8, 0, 28],
    [7.7, 11, 17, 39],
    [25, 9, 49, 6.89]
], dtype = float)

print('\n', B[1:3, 1:3])    # Central 2x2 block of the matrix

# Leave the second index of i:n unspecified mean take all the columns/rows from the i-th one, leaving the fisrt unspecified means take all the 
# rows/columns up to the n-1.
# That's all about entries exctraction

print('\n', B[1:3, 0:3])




(3, 3)
3.0
[1. 3. 6.]
[3. 3. 9.]
[[1. 3. 6.]
 [4. 3. 8.]]
[[1. 3.]
 [4. 3.]
 [5. 9.]]
[[8.]
 [0.]]

 [[1. 3.]
 [4. 3.]]

 [[ 8.  0.]
 [11. 17.]]

 [[ 6.   8.   0. ]
 [ 7.7 11.  17. ]]


In [18]:
# Now some more advance entries extraction commands

print(B[::2, :])    # Extract all the rows jumping from 0-th to 2-nd to 4-th etc... basically skipping one index each time, starting from 0

print('\n', B[1::2, :]) # Here we start from the 1-st row, which means print all the odd-indiced rows, the command above exctrats the even-indeced ones

print('\n', B[:, ::3])  # The same can be done with the columns too, obviously the specified "jump" it is not strickly 2, it can be any int numbers

print('\n', B[:, ::4])  # If we goes out of bounds (B has j = 0,1,2,3) the command prints only the possible entries, in this case only the first column

# In this way it is obviously possible two exctract any set of entries, ordered as you wish, from the matrix

# if we want to exactly speficy rows an columns we can pass a list:

print('\n', B[[1,3], [0,2]])    # this extract the element in 1,0 and in 3,2

print('\n', B[:, [2,3]])    # It prints all the entries of columns 2 and 3

print('\n', B[[0,1], :])    # First two rows, that can be done also with B[:2, :]

# To select every intersection of rows i, n and columns j, m use np.ix_

print('\n', B[np.ix_([0,2], [1,3])])



[[ 1.   2.4 -4.   5.5]
 [ 7.7 11.  17.  39. ]]

 [[ 6.    8.    0.   28.  ]
 [25.    9.   49.    6.89]]

 [[ 1.    5.5 ]
 [ 6.   28.  ]
 [ 7.7  39.  ]
 [25.    6.89]]

 [[ 1. ]
 [ 6. ]
 [ 7.7]
 [25. ]]

 [ 6. 49.]

 [[-4.    5.5 ]
 [ 0.   28.  ]
 [17.   39.  ]
 [49.    6.89]]

 [[ 1.   2.4 -4.   5.5]
 [ 6.   8.   0.  28. ]]

 [[ 2.4  5.5]
 [11.  39. ]]


In [29]:
# Boolean masks

print(B[B > 1]) # To select individual element of B that respect a specific condition. Returns a 1d array

print('\n', B[(B>1) | (B%2 != 0)])  # You can do whatever you want, but REMEMBER THE PARENTESIS (instead of 'or' use | and & for 'and', the bitwise
                                    # logical operation basically)

# Then logical masks on specific rows and columns

print('\n', B[B[:, 3]>25, :])    # Rows whose third entries exceed 30

print('\n', B[:, B[0,:]<=1])    # Columns whose first entries are at lest 1

#NOTE: After selecting a specific set of elements of an array, is it possible to overwrite them: B[:, B[0,:]<=1] = 0, put to zero all the selected elements
# A[:,1] = 0 set the selected column to zero.

[ 2.4   5.5   6.    8.   28.    7.7  11.   17.   39.   25.    9.   49.
  6.89]

 [ 1.    2.4   5.5   6.    8.   28.    7.7  11.   17.   39.   25.    9.
 49.    6.89]

 [[ 6.   8.   0.  28. ]
 [ 7.7 11.  17.  39. ]]

 [[ 1.  -4. ]
 [ 6.   0. ]
 [ 7.7 17. ]
 [25.  49. ]]


In [ ]:
# Keeping or removing an axis

print(B[1, :].shape)    # Gives the shape of the array obtain exctracing the second row (1d array of 4 entries)

print('\n', B[1:3, :].shape)    # Gives the shape of the array composed by the second and the third rows of B (2x4 matrix)

print('\n', B[:, 2:3].shape)    # Same thing can be done with columns

# An alternative idea of this operation is that give an interger index is equivalent to remove an axis, while giving a slice (n:m) keeps that axis.
# But it is kinda tautologic


(4,)

 (2, 4)

 (4, 1)


In [30]:
# Tecnical detail: ordinary slices share data with the original array, so to have an indipend quantity (like extracting an array from a bigger one and
# operate on it independently from the original one) we use:

row = A[0,:]
row = A[0,:].copy() # Acompletely independent object from the row in the line above

In [ ]:
# Useful operation for gauss elimination:

A[1,:] -= 5.*A[0,:] # which multiply by a factor 5 the first row and subtrac it to the second one

In [ ]:
# Reshape:

print(A.reshape(1,3,3)) # It change the dimensions of an array. You can do it if the total number of elements needed to fill the new array is
                        # equal to the number of elements in the original array

# nice trick
print('\n', A.reshape(-1))  # Reduce the number of dimensions of -1

A.reshape(-1).copy()

print('\n', A.reshape(-1, 1))   # (9,1) column 

print('\n', A.reshape(1, -1))   # (1,9) row

# usefull for transpose 1d array in lin. sys. when needed. It DOES NOT TRANSPOSE MATRCES

[[[1. 3. 6.]
  [4. 3. 8.]
  [5. 9. 0.]]]

 [1. 3. 6. 4. 3. 8. 5. 9. 0.]

 [[1.]
 [3.]
 [6.]
 [4.]
 [3.]
 [8.]
 [5.]
 [9.]
 [0.]]

 [[1. 3. 6. 4. 3. 8. 5. 9. 0.]]


## GUASS ELIMANTION.

Now we can actually start experimenting with proper Gauss elimination

In [16]:
import numpy as np

A = np.array([
    [ 2,  3,  1,  5,  4],
    [ 7,  1,  4,  2,  6],
    [-9,  2,  5,  1,  3],
    [ 4, -8,  2,  6,  1],
    [ 1,  5, -7,  3,  9]
], dtype=float)

print(A)
print(A[:,0].shape[0])

B = A.copy()

for i in range(A[:,0].shape[0]-1):
    # Let's start ordering the columns
    where_max = i + np.argmax(abs(A[i:, i]))
    row = A[i, :].copy()
    A[i,:] = A[where_max, :]
    A[where_max, :] = row
    # Now the rows should have been exchange one with other

G = A.copy()

print('\n',A)



[[ 2.  3.  1.  5.  4.]
 [ 7.  1.  4.  2.  6.]
 [-9.  2.  5.  1.  3.]
 [ 4. -8.  2.  6.  1.]
 [ 1.  5. -7.  3.  9.]]
5

 [[-9.  2.  5.  1.  3.]
 [ 4. -8.  2.  6.  1.]
 [ 1.  5. -7.  3.  9.]
 [ 2.  3.  1.  5.  4.]
 [ 7.  1.  4.  2.  6.]]


In [ ]:
#even better for the row swap
print(B.shape, '\n', B.shape[0])

for i in range(B.shape[0]-1):
    # Let's start ordering the columns
    where_max = i + np.argmax(abs(B[i:, i]))
    B[[i, where_max], :] = B[[where_max, i], :]
    # Now the rows should have been exchange one with other

print('\n',B)

G.shape

print(G != B)

# This version correctly work


(5, 5) 
 5

 [[-9.  2.  5.  1.  3.]
 [ 4. -8.  2.  6.  1.]
 [ 1.  5. -7.  3.  9.]
 [ 2.  3.  1.  5.  4.]
 [ 7.  1.  4.  2.  6.]]
[[False False False False False]
 [False False False False False]
 [False False False False False]
 [False False False False False]
 [False False False False False]]
